# 30 septembre — Tri par insertion

[Notes et preuves](note.md#12-suite--le-tri-par-insertion) · [TD 2, exercice 2, question 4](../23.09/TD2_original.pdf)

![Insertions et décalages](tri_insertion.svg)

Le tableau `[4, 2, 3, 5, 1]` vient de la photo. Le code reprend la méthode projetée en ajoutant la condition nécessaire `j > 0`. Il modifie la liste sur place et renvoie implicitement `None`, conformément à la convention du professeur.

In [1]:
def tri_insertion(tab):
    for i in range(1, len(tab)):
        val = tab[i]
        j = i
        while j > 0 and tab[j - 1] > val:
            tab[j] = tab[j - 1]
            j = j - 1
        tab[j] = val


## Un appel qui modifie la liste

On appelle `tri_insertion(tab)`, puis on consulte `tab`. Écrire `tab = tri_insertion(tab)` ferait perdre la référence à la liste dans cette variable, car le résultat renvoyé est `None`.

In [2]:
exemple = [4, 2, 3, 5, 1]
retour = tri_insertion(exemple)
assert retour is None
assert exemple == [1, 2, 3, 4, 5]
print("Liste après le tri :", exemple)
print("Valeur renvoyée :", retour)


Liste après le tri : [1, 2, 3, 4, 5]
Valeur renvoyée : None


## Suivre les insertions et les décalages

Cette cellule reprend le tri avec des affichages et des copies pour conserver les étapes. Ces copies appartiennent à l'outil de suivi, pas à l'algorithme de tri en espace constant.

Le préfixe vert du schéma est trié mais pas définitivement placé : l'insertion finale de 1 décale toutes ses valeurs.

In [3]:
tab = [4, 2, 3, 5, 1]
etapes = [tab.copy()]
for i in range(1, len(tab)):
    val = tab[i]
    j = i
    print(f"i={i}, val={val}")
    while j > 0 and tab[j - 1] > val:
        print(f"  Décaler {tab[j - 1]} de la case {j - 1} vers {j}")
        tab[j] = tab[j - 1]
        j -= 1
    tab[j] = val
    etapes.append(tab.copy())
    print(f"  Placer {val} en {j} : {tab}")

assert etapes == [
    [4, 2, 3, 5, 1],
    [2, 4, 3, 5, 1],
    [2, 3, 4, 5, 1],
    [2, 3, 4, 5, 1],
    [1, 2, 3, 4, 5],
]


i=1, val=2
  Décaler 4 de la case 0 vers 1
  Placer 2 en 0 : [2, 4, 3, 5, 1]
i=2, val=3
  Décaler 4 de la case 1 vers 2
  Placer 3 en 1 : [2, 3, 4, 5, 1]
i=3, val=5
  Placer 5 en 3 : [2, 3, 4, 5, 1]
i=4, val=1
  Décaler 5 de la case 3 vers 4
  Décaler 4 de la case 2 vers 3
  Décaler 3 de la case 1 vers 2
  Décaler 2 de la case 0 vers 1
  Placer 1 en 0 : [1, 2, 3, 4, 5]


## Comprendre le trou logique

La case `j` n'est pas réellement supprimée : on ignore temporairement son contenu. La valeur à insérer est sauvegardée dans `val`. Après un décalage, des doublons apparaissent physiquement dans la liste, sans perte de la valeur sauvegardée.

Dans le schéma, le point remplace cette case ignorée. La cellule suivante affiche à la fois le tableau physique et la vue logique de la dernière insertion.

In [4]:
tab = [2, 3, 4, 5, 1]
val = tab[4]
j = 4
zoom = []
while True:
    vue = tab.copy()
    vue[j] = None  # None représente ici le trou du dessin seulement.
    zoom.append(vue)
    print("Liste physique :", tab, "; trou logique en", j, "; val =", val)
    if not (j > 0 and tab[j - 1] > val):
        break
    tab[j] = tab[j - 1]
    j -= 1
tab[j] = val
zoom.append(tab.copy())
assert tab == [1, 2, 3, 4, 5]
print("Après insertion :", tab)


Liste physique : [2, 3, 4, 5, 1] ; trou logique en 4 ; val = 1
Liste physique : [2, 3, 4, 5, 5] ; trou logique en 3 ; val = 1
Liste physique : [2, 3, 4, 4, 5] ; trou logique en 2 ; val = 1
Liste physique : [2, 3, 3, 4, 5] ; trou logique en 1 ; val = 1
Liste physique : [2, 2, 3, 4, 5] ; trou logique en 0 ; val = 1
Après insertion : [1, 2, 3, 4, 5]


## Preuve et coût à retenir

Avant le tour `i`, le préfixe `tab[:i]` est trié et conserve les valeurs initiales de ce préfixe. L'insertion agrandit cette zone triée d'une case.

Dans la boucle intérieure, ignorer la case `j` laisse les anciennes valeurs du préfixe dans leur ordre ; celles déjà décalées sont supérieures à `val`. À l'arrêt, `j == 0` ou le voisin gauche est inférieur ou égal à `val`. L'affectation finale replace `val` entre des voisins compatibles.

Le variant intérieur est `j`, qui diminue et reste naturel grâce au test de borne. Temps Θ(n) sur une liste déjà triée, Θ(n²) dans le pire cas, espace auxiliaire O(1). La comparaison stricte `>` préserve l'ordre des éléments de même clé : le tri est stable.

## Vérifications

Les tests couvrent les listes vides, les singletons, les valeurs négatives, les doublons et tous les ordres de petites listes. Un test séparé vérifie la stabilité sur des objets identifiés.

In [5]:
from itertools import product

nombre = 0
for taille in range(6):
    for valeurs in product((-1, 0, 1), repeat=taille):
        tab = list(valeurs)
        retour = tri_insertion(tab)
        assert retour is None
        assert tab == sorted(valeurs)
        nombre += 1

class Element:
    def __init__(self, cle, nom):
        self.cle = cle
        self.nom = nom

    def __gt__(self, autre):
        return self.cle > autre.cle

objets = [Element(2, "A"), Element(1, "B"), Element(2, "C"), Element(1, "D")]
tri_insertion(objets)
assert [x.nom for x in objets] == ["B", "D", "A", "C"]
print(nombre, "tris vérifiés ; stabilité et retour None vérifiés.")


364 tris vérifiés ; stabilité et retour None vérifiés.
